In [1]:
#word

# Multiple word with occurences
words_count = [("hug", 10), ("pug", 5), ("pun", 12), ("bun", 4), ("hugs", 5)]

words = []
for word, occurences in words_count:
    words.extend([word] * occurences)


In [2]:
# Get initial vocab
initial_vocab = list()
for word in words:
    for char in word:
        initial_vocab.append(char)

In [3]:
VOCAB = initial_vocab

In [4]:
def get_subwords(word):
    if len(word) != 3:
        word_1 = word[:-1]
        word_2 = word[1:]
        VOCAB.append(word_1)
        VOCAB.append(word_2)
        return get_subwords(word_1) + get_subwords(word_2)
    else:
        return [word[1:], word[:-1]]

In [5]:
for word in words:
    subwords = get_subwords(word)
    for subword in subwords:
        VOCAB.append(subword)

In [6]:
import collections
counter = collections.Counter(VOCAB)

In [7]:
counter

Counter({'u': 36,
         'ug': 25,
         'g': 20,
         'p': 17,
         'pu': 17,
         'n': 16,
         'un': 16,
         'h': 15,
         'hu': 15,
         's': 5,
         'hug': 5,
         'ugs': 5,
         'gs': 5,
         'b': 4,
         'bu': 4})

In [8]:
distinct_words = set(words)

In [9]:
distinct_words

{'bun', 'hug', 'hugs', 'pug', 'pun'}

In [10]:
bun_word = list(distinct_words)[0]

In [11]:
dict(counter)

{'h': 15,
 'u': 36,
 'g': 20,
 'p': 17,
 'n': 16,
 'b': 4,
 's': 5,
 'ug': 25,
 'hu': 15,
 'pu': 17,
 'un': 16,
 'bu': 4,
 'hug': 5,
 'ugs': 5,
 'gs': 5}

In [12]:
bun_word

'hug'

In [13]:
corpus = []
for key, value in counter.items():
    if key in bun_word:
        corpus.append({key: value})

In [14]:
corpus

[{'h': 15}, {'u': 36}, {'g': 20}, {'ug': 25}, {'hu': 15}, {'hug': 5}]

In [15]:
from itertools import combinations, permutations

In [16]:
parts = [ key for el in corpus for key, value in el.items()]

In [17]:
parts

['h', 'u', 'g', 'ug', 'hu', 'hug']

In [18]:
matches = []
for r in range(1, len(parts) + 1):
    for combo in combinations(parts, r):
        for perm in permutations(combo):
            if ''.join(perm) == bun_word:
                matches.append(perm)

In [19]:
matches_corpus = []
for match in matches:
    match_dict = {}
    for part in match:
        match_dict[part] = counter[part]
    matches_corpus.append(match_dict)

In [20]:
matches_corpus

[{'hug': 5},
 {'h': 15, 'ug': 25},
 {'hu': 15, 'g': 20},
 {'h': 15, 'u': 36, 'g': 20}]

In [21]:
sum_counter = 0
for key, value in counter.items():
    sum_counter += value
sum_counter

205

In [22]:
sum_counter

205

In [23]:
match_sums = []
for match in matches_corpus:
    match_sum = 1
    for key, value in match.items():
        div = value / sum_counter
        match_sum *= div
    match_sums.append(dict({match_sum: match}))
print(match_sums)

# Select max match value and corresponding match
max_match = max(match_sums, key=lambda x: list(x.keys())[0])
print(max_match)


[{0.024390243902439025: {'hug': 5}}, {0.00892325996430696: {'h': 15, 'ug': 25}}, {0.007138607971445568: {'hu': 15, 'g': 20}}, {0.0012536092047416606: {'h': 15, 'u': 36, 'g': 20}}]
{0.024390243902439025: {'hug': 5}}


## Loss during training

In [24]:
# Corpus
corpus = [("hug", 10), ("pug", 5), ("pun", 12), ("bun", 4), ("hugs", 5)]

In [ ]:
# Prerequisites
## Let's say we already done the tokenization and scoring for words

"hug": ["hug"] (score 0.071428)
"pug": ["pu", "g"] (score 0.007710)
"pun": ["pu", "n"] (score 0.006168)
"bun": ["bu", "n"] (score 0.001451)
"hugs": ["hug", "s"] (score 0.001701)

In [25]:
from math import log

In [26]:
loss = 10 * (-log(0.071428)) + 5 * (-log(0.007710)) + 12 * (-log(0.006168)) + 4 * (-log(0.001451)) + 5 * (-log(0.001701))
loss

169.8021105143228

In [ ]:
## if we remove hug token with hug and hugs it's become
"hug": ["hu", "g"] (score 0.006802)
"hugs": ["hu", "gs"] (score 0.001701)

In [27]:
loss_a = (- 10 * (-log(0.071428)) + 10 * (-log(0.006802)))
loss_a


23.51473262749874

In [28]:
# So loss 
loss + loss_a

193.31684314182155

## Implementing Unigram


In [29]:
corpus = [
    "This is the Hugging Face Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]

In [30]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("xlnet-base-cased")

config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [31]:
from collections import defaultdict

word_freqs = defaultdict(int)

In [32]:
word_freqs

defaultdict(int, {})

In [33]:
for text in corpus:
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    new_words = [word for word, offset in words_with_offsets]
    for word in new_words:
        word_freqs[word] += 1

In [34]:
word_freqs

defaultdict(int,
            {'▁This': 3,
             '▁is': 2,
             '▁the': 1,
             '▁Hugging': 1,
             '▁Face': 1,
             '▁Course.': 1,
             '▁chapter': 1,
             '▁about': 1,
             '▁tokenization.': 1,
             '▁section': 1,
             '▁shows': 1,
             '▁several': 1,
             '▁tokenizer': 1,
             '▁algorithms.': 1,
             '▁Hopefully,': 1,
             '▁you': 1,
             '▁will': 1,
             '▁be': 1,
             '▁able': 1,
             '▁to': 1,
             '▁understand': 1,
             '▁how': 1,
             '▁they': 1,
             '▁are': 1,
             '▁trained': 1,
             '▁and': 1,
             '▁generate': 1,
             '▁tokens.': 1})

In [ ]:
# Calculate the frequency of characters and subwords

char_freqs = defaultdict(int)
subwords_freqs = defaultdict(int)
for word, freq in word_freqs.items():
    for i in range(len(word)):
        char_freqs[word[i]] += freq
        # Loop through the subwords of length at least 2
        for j in range(i + 2, len(word) + 1):
            subwords_freqs[word[i:j]] += freq


In [36]:
char_freqs

defaultdict(int,
            {'▁': 31,
             'T': 3,
             'h': 9,
             'i': 13,
             's': 13,
             't': 14,
             'e': 21,
             'H': 2,
             'u': 6,
             'g': 5,
             'n': 11,
             'F': 1,
             'a': 12,
             'c': 3,
             'C': 1,
             'o': 13,
             'r': 9,
             '.': 4,
             'p': 2,
             'b': 3,
             'k': 3,
             'z': 2,
             'w': 3,
             'v': 1,
             'l': 7,
             'm': 1,
             'f': 1,
             'y': 3,
             ',': 1,
             'd': 4})

In [37]:
subwords_freqs

defaultdict(int,
            {'▁T': 3,
             '▁Th': 3,
             '▁Thi': 3,
             '▁This': 3,
             'Th': 3,
             'Thi': 3,
             'This': 3,
             'hi': 3,
             'his': 3,
             'is': 5,
             '▁i': 2,
             '▁is': 2,
             '▁t': 7,
             '▁th': 2,
             '▁the': 2,
             'th': 3,
             'the': 2,
             'he': 2,
             '▁H': 2,
             '▁Hu': 1,
             '▁Hug': 1,
             '▁Hugg': 1,
             '▁Huggi': 1,
             '▁Huggin': 1,
             '▁Hugging': 1,
             'Hu': 1,
             'Hug': 1,
             'Hugg': 1,
             'Huggi': 1,
             'Huggin': 1,
             'Hugging': 1,
             'ug': 1,
             'ugg': 1,
             'uggi': 1,
             'uggin': 1,
             'ugging': 1,
             'gg': 1,
             'ggi': 1,
             'ggin': 1,
             'gging': 1,
             'gi': 1,
             '

In [38]:
# Sort subwords by frequency
sorted_subwords = sorted(subwords_freqs.items(), key=lambda x: x[1], reverse=True)
sorted_subwords[:10]

[('▁t', 7),
 ('is', 5),
 ('er', 5),
 ('▁a', 5),
 ('▁to', 4),
 ('to', 4),
 ('en', 4),
 ('▁T', 3),
 ('▁Th', 3),
 ('▁Thi', 3)]

In [ ]:
# Create vocabulary of size 300
token_freqs = list(char_freqs.items()) + sorted_subwords[: 300 - len(char_freqs)]
token_freqs = {token: freq for token, freq in token_freqs}

In [40]:
token_freqs

{'▁': 31,
 'T': 3,
 'h': 9,
 'i': 13,
 's': 13,
 't': 14,
 'e': 21,
 'H': 2,
 'u': 6,
 'g': 5,
 'n': 11,
 'F': 1,
 'a': 12,
 'c': 3,
 'C': 1,
 'o': 13,
 'r': 9,
 '.': 4,
 'p': 2,
 'b': 3,
 'k': 3,
 'z': 2,
 'w': 3,
 'v': 1,
 'l': 7,
 'm': 1,
 'f': 1,
 'y': 3,
 ',': 1,
 'd': 4,
 '▁t': 7,
 'is': 5,
 'er': 5,
 '▁a': 5,
 '▁to': 4,
 'to': 4,
 'en': 4,
 '▁T': 3,
 '▁Th': 3,
 '▁Thi': 3,
 '▁This': 3,
 'Th': 3,
 'Thi': 3,
 'This': 3,
 'hi': 3,
 'his': 3,
 'th': 3,
 'ou': 3,
 'se': 3,
 '▁tok': 3,
 '▁toke': 3,
 '▁token': 3,
 'tok': 3,
 'toke': 3,
 'token': 3,
 'ok': 3,
 'oke': 3,
 'oken': 3,
 'ke': 3,
 'ken': 3,
 '▁s': 3,
 'ra': 3,
 'nd': 3,
 '▁i': 2,
 '▁is': 2,
 '▁th': 2,
 '▁the': 2,
 'the': 2,
 'he': 2,
 '▁H': 2,
 'in': 2,
 'rs': 2,
 'te': 2,
 '▁ab': 2,
 'ab': 2,
 '▁tokeni': 2,
 '▁tokeniz': 2,
 'tokeni': 2,
 'tokeniz': 2,
 'okeni': 2,
 'okeniz': 2,
 'keni': 2,
 'keniz': 2,
 'eni': 2,
 'eniz': 2,
 'ni': 2,
 'niz': 2,
 'iz': 2,
 'at': 2,
 'ti': 2,
 'tio': 2,
 'tion': 2,
 'io': 2,
 'ion': 2,
 'on':

In [ ]:

from math import log
# model  will store the logarithms of the probabilities, because it’s more numerically stable to add logarithms than to multiply small numbers
total_sum = sum([freq for token, freq in token_freqs.items()])
model = {token: -log(freq / total_sum) for token, freq in token_freqs.items()}

In [42]:
total_sum

594

In [43]:
model

{'▁': 2.952892114877499,
 'T': 5.288267030694535,
 'h': 4.189654742026425,
 'i': 3.821929961901108,
 's': 3.821929961901108,
 't': 3.7478219897473863,
 'e': 3.342356881639222,
 'H': 5.6937321388027,
 'u': 4.59511985013459,
 'g': 4.777441406928545,
 'n': 3.9889840465642745,
 'F': 6.386879319362645,
 'a': 3.9019726695746444,
 'c': 5.288267030694535,
 'C': 6.386879319362645,
 'o': 3.821929961901108,
 'r': 4.189654742026425,
 '.': 5.000584958242754,
 'p': 5.6937321388027,
 'b': 5.288267030694535,
 'k': 5.288267030694535,
 'z': 5.6937321388027,
 'w': 5.288267030694535,
 'v': 6.386879319362645,
 'l': 4.440969170307332,
 'm': 6.386879319362645,
 'f': 6.386879319362645,
 'y': 5.288267030694535,
 ',': 6.386879319362645,
 'd': 5.000584958242754,
 '▁t': 4.440969170307332,
 'is': 4.777441406928545,
 'er': 4.777441406928545,
 '▁a': 4.777441406928545,
 '▁to': 5.000584958242754,
 'to': 5.000584958242754,
 'en': 5.000584958242754,
 '▁T': 5.288267030694535,
 '▁Th': 5.288267030694535,
 '▁Thi': 5.2882670

In [44]:
def encode_word(word, model):
    best_segmentations = [{"start": 0, "score": 1}] + [
        {"start": None, "score": None} for _ in range(len(word))
    ]
    for start_idx in range(len(word)):
        # This should be properly filled by the previous steps of the loop
        best_score_at_start = best_segmentations[start_idx]["score"]
        for end_idx in range(start_idx + 1, len(word) + 1):
            token = word[start_idx:end_idx]
            if token in model and best_score_at_start is not None:
                score = model[token] + best_score_at_start
                # If we have found a better segmentation ending at end_idx, we update
                if (
                    best_segmentations[end_idx]["score"] is None
                    or best_segmentations[end_idx]["score"] > score
                ):
                    best_segmentations[end_idx] = {"start": start_idx, "score": score}

    segmentation = best_segmentations[-1]
    if segmentation["score"] is None:
        # We did not find a tokenization of the word -> unknown
        return ["<unk>"], None

    score = segmentation["score"]
    start = segmentation["start"]
    end = len(word)
    tokens = []
    while start != 0:
        tokens.insert(0, word[start:end])
        next_start = best_segmentations[start]["start"]
        end = start
        start = next_start
    tokens.insert(0, word[start:end])
    return tokens, score

In [45]:
print(encode_word("Hopefully", model))
print(encode_word("This", model))

(['H', 'o', 'p', 'e', 'f', 'u', 'll', 'y'], 41.5157494601402)
(['This'], 6.288267030694535)


In [46]:
def compute_loss(model):
    loss = 0
    for word, freq in word_freqs.items():
        _, word_loss = encode_word(word, model)
        loss += freq * word_loss
    return loss

In [47]:
compute_loss(model)


413.10377642940875

In [ ]:
import copy


def compute_scores(model):
    scores = {}
    model_loss = compute_loss(model)
    for token, score in model.items():
        # We always keep tokens of length 1
        if len(token) == 1:
            continue
        model_without_token = copy.deepcopy(model)
        _ = model_without_token.pop(token)
        scores[token] = compute_loss(model_without_token) - model_loss
    return scores

In [48]:
import copy


def compute_scores(model):
    scores = {}
    model_loss = compute_loss(model)
    for token, score in model.items():
        # We always keep tokens of length 1
        if len(token) == 1:
            continue
        model_without_token = copy.deepcopy(model)
        _ = model_without_token.pop(token)
        scores[token] = compute_loss(model_without_token) - model_loss
    return scores

In [49]:
scores = compute_scores(model)
print(scores["ll"])
print(scores["his"])

6.376412403623874
0.0


In [50]:
percent_to_remove = 0.1
while len(model) > 100:
    scores = compute_scores(model)
    sorted_scores = sorted(scores.items(), key=lambda x: x[1])
    # Remove percent_to_remove tokens with the lowest scores.
    for i in range(int(len(model) * percent_to_remove)):
        _ = token_freqs.pop(sorted_scores[i][0])

    total_sum = sum([freq for token, freq in token_freqs.items()])
    model = {token: -log(freq / total_sum) for token, freq in token_freqs.items()}

In [51]:
def tokenize(text, model):
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    pre_tokenized_text = [word for word, offset in words_with_offsets]
    encoded_words = [encode_word(word, model)[0] for word in pre_tokenized_text]
    return sum(encoded_words, [])


In [52]:
tokenize("This is the Hugging Face course.", model)

['▁This',
 '▁is',
 '▁the',
 '▁Hugging',
 '▁Face',
 '▁',
 'c',
 'ou',
 'r',
 's',
 'e',
 '.']